In [ ]:
from hogsboutils import cleaning_mail, read_contacts, read_schedule, email_auth
import pandas as pd
import numpy as np
import datetime

In [ ]:
contacts = read_contacts(fetch_sheet=True)
contacts

In [ ]:
schedule, areas_se, areas_en= read_schedule("2026-09-04")
schedule

In [ ]:
nested = schedule[['a1', 'a2', 'a3', 'a4']].values
flat_list = [item for sublist in nested for item in sublist]
shifts = [item for item in flat_list if type(item) is str]
shifts = [item for item in shifts if 'ALL' not in item]

names = list(np.unique(shifts))

### Shifts per person

In [ ]:
from collections import Counter
totals = Counter(shifts)

In [ ]:
for person, num in totals.items():
    if num < 2:
        print(f'{person}\t {num}')

In [ ]:
for name in names:
    if '?' in name:
        continue
    if name not in contacts['Cleaning name'].values:
        print("error, contact not found for", name)

### Missing from cleaning schedule

In [ ]:
contacts['Cleaning name'] = contacts['Cleaning name'].astype(str)
missing = contacts[contacts['Cleaning name'] == 'nan']['Namn / Name'].values
not_scheduled = set(contacts['Cleaning name'].values).difference(names)
for name in missing:
    print(name, "No cleaning name in contacts")
print('')
for name in not_scheduled:
    print(name, "Not sheduled for cleaning")

### Test run

In [ ]:
def weekly_clean_send(df, week, send=False, print_msg=True):
    cleaners = df[df.date==week][['a1', 'a2', 'a3', 'a4']].iloc[0].to_dict()
    for area, cleaner in cleaners.items():
        if area =='date':
            continue
        if 'HELA HUSET' in str(cleaner):
            return
        cleaner_row = contacts[contacts['Cleaning name'] == cleaner]
        if len(cleaner_row) == 0:
            print(f"FAIL! for {cleaner} not found in contacts sheet")
            continue
        if len(cleaner_row) > 1:
            if '&' not in cleaner:
                print(f"FAIL! Found unexpected multiple matches for {cleaner} {len(cleaner_row)}")
        for mail in cleaner_row['E-post / Email Address']:
            if send:
                cleaning_mail(cleaner, mail, area)
            if print_msg:
                print(week, cleaner, mail, areas_en[area])


for this_date in schedule.date:
    weekly_clean_send(schedule, this_date, print_msg=False)

In [ ]:
row_number = np.abs(schedule['send_date'] - datetime.datetime.now()).argmin()
this_date = schedule.iloc[row_number]['date']
schedule.iloc[row_number]

In [ ]:
weekly_clean_send(schedule, this_date, print_msg=True)

In [ ]:
contacts['Cleaning name'] = contacts['Cleaning name'].astype(str)
missing = contacts[contacts['Cleaning name'] == 'nan']['Namn / Name'].values
not_scheduled = set(contacts['Cleaning name'].values).difference(names)
for name in missing:
    print(name, "No cleaning name in contacts")

for name in not_scheduled:
    print(name, "Not sheduled for cleaning")